In [4]:
from langchain_google_genai import GoogleGenerativeAI
from langchain_google_genai import GoogleGenerativeAIEmbeddings
from langchain_chroma import Chroma
import os

from langchain_core.prompts import PromptTemplate
from langchain_core.documents import Document
from langchain.document_loaders import WebBaseLoader, PyPDFLoader, TextLoader, CSVLoader, UnstructuredFileLoader, Docx2txtLoader, UnstructuredPowerPointLoader, UnstructuredURLLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter

from pydantic import BaseModel
from typing import Optional

def read_text_file(file_path):

    with open(file_path, 'r') as file:
        return file.read()

class TextRewriterArgs(BaseModel):
    input_text: str
    file_url: str
    file_type: str
    rewrite_instructions: str
    lang: Optional[str] = "en"

/home/bipin/Documents/Reality AI Lab/marvel-ai-backend/app/env/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
USER_AGENT environment variable not set, consider setting it to identify your requests.


In [5]:
input_text = "The quick brown fox jumped over the lazy dog."
file_url = "None"
file_type = "None"
rewrite_instructions = "Simplify this sentence"

prompt_template_text = read_text_file("prompt/text-rewriter-prompt.txt")

args = TextRewriterArgs(
            input_text=input_text,
            file_url=file_url,
            file_type=file_type,
            rewrite_instructions=rewrite_instructions,
            lang='en'
        )

prompt_template = PromptTemplate(template=prompt_template_text,
                                 input_variables=["context", "rewrite_instructions", "language"])

if args.file_type == 'txt':
    loader = TextLoader("sample1.txt", encoding = "UTF-8")
    content = loader.load()[0].page_content

elif args.file_type == 'pdf':
    loader = PyPDFLoader("sample2.pdf")
    content = [doc.page_content for doc in loader.load()]
    content = " ".join(content)

elif args.file_type == 'docx':
    loader = UnstructuredFileLoader("sample3.docx")
    content = [doc.page_content for doc in loader.load()]
    content = " ".join(content)

elif args.file_type == 'pptx':
    loader = UnstructuredPowerPointLoader("sample4.ppt")
    
    
content = input_text

prompt = prompt_template.format(
            context = content,
            rewrite_instructions = args.rewrite_instructions,
            language = args.lang)


In [12]:
!pwd

/home/bipin/Documents/Reality AI Lab/marvel-ai-backend/app/tools/text_rewriter


In [18]:
import os
from dotenv import load_dotenv, find_dotenv

load_dotenv(find_dotenv())
dotenv_path = os.path.join(os.path.dirname(__file__), "../../../.env/.env")
api_key = os.getenv("GOOGLE_API_KEY")
os.environ["GOOGLE_API_KEY"] = api_key

print(f"Loaded API Key: {api_key}")  # Debugging step

TypeError: str expected, not NoneType

In [21]:
os.path.dirname(__file__)

NameError: name '__file__' is not defined

In [7]:
llm = GoogleGenerativeAI(model="gemini-1.5-flash")

In [8]:
llm.invoke(prompt)

DefaultCredentialsError: 
  No API_KEY or ADC found. Please either:
    - Set the `GOOGLE_API_KEY` environment variable.
    - Manually pass the key with `genai.configure(api_key=my_api_key)`.
    - Or set up Application Default Credentials, see https://ai.google.dev/gemini-api/docs/oauth for more information.

In [3]:
import requests

url = "http://127.0.0.1:8000/test-rewriter"
data = {
    "input_text": "The quick brown fox jumps over the lazy dog.",
    "rewrite_instructions": "Summarize the text.",
    "file_url": "None",
    "file_type": "None",
    "lang": "en"
}

response = requests.post(url, json=data)
print(response.json())


{'detail': 'Error: Failed to generate Re-written Text: Your default credentials were not found. To set up Application Default Credentials, see https://cloud.google.com/docs/authentication/external/set-up-adc for more information.'}
